## **RAG의 핵심, 문서 검색기 Retriever**

### **사용자의 쿼리를 재해석하여 검색하다, MultiQueryRetriever**

**Chroma DB에 문서 벡터 저장**

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
#헌법 PDF 파일 로드
loader = PyPDFLoader(r"대한민국 헌법.pdf")
pages = loader.load_and_split()

#PDF 파일을 500자 청크로 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(pages)

#ChromaDB에 청크들을 벡터 임베딩으로 저장(OpenAI 임베딩 모델 활용)
db = Chroma.from_documents(docs, OpenAIEmbeddings(model = 'text-embedding-3-small'))

**질문을 여러 버전으로 재해석하여 Retriever에 활용**

In [6]:
#```Chroma DB에 대한민국 헌법 PDF 임베딩 변환 및 저장하는 과정은 위 셀에 있습니다```
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

#질문 문장 question으로 저장
question = "국회의원의 의무는 무엇이 있나요?"
#여러 버전의 질문으로 변환하는 역할을 맡을 LLM 선언
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0125",
                 temperature = 0)
#MultiQueryRetriever에 벡터DB 기반 Retriever와 LLM 선언
retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(), llm=llm
)

# 여러 버전의 문장 생성 결과를 확인하기 위한 로깅 과정
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

#여러 버전 질문 생성 결과와 유사 청크 검색 개수 출력
unique_docs = retriever_from_llm.invoke(input=question)
len(unique_docs)

5

In [7]:
unique_docs

[Document(metadata={'moddate': '2017-08-06T23:09:57+09:00', 'producer': 'iText 2.1.7 by 1T3XT', 'source': '대한민국 헌법.pdf', 'creationdate': '2017-08-06T23:09:57+09:00', 'total_pages': 21, 'page_label': '8', 'creator': 'PyPDF', 'page': 7}, page_content='제43조 국회의원은 법률이 정하는 직을 겸할 수 없다.\n \n제44조 ①국회의원은 현행범인인 경우를 제외하고는 회기중 국회의 동의없이 체포 또는 구금되\n지 아니한다.\n②국회의원이  회기전에  체포  또는  구금된  때에는  현행범인이  아닌  한  국회의  요구가  있으면\n회기중  석방된다.\n \n제45조 국회의원은 국회에서 직무상 행한 발언과 표결에 관하여 국회외에서 책임을 지지 아니한\n다.\n \n제46조 ①국회의원은 청렴의 의무가 있다.\n②국회의원은 국가이익을 우선하여 양심에 따라 직무를 행한다.\n③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하\n여 재산상의 권리ㆍ이익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.\n \n제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의 임시회는 대\n통령 또는 국회재적의원 4분의 1 이상의 요구에 의하여 집회된다.'),
 Document(metadata={'total_pages': 21, 'creator': 'PyPDF', 'source': '대한민국 헌법.pdf', 'creationdate': '2017-08-06T23:09:57+09:00', 'page': 9, 'producer': 'iText 2.1.7 by 1T3XT', 'moddate': '2017-08-06T23:09:57+09:00', 'page_label': '10'}, p

### **컨텍스트 재정렬, Long-Context Reorder**

**[Long-Context Reorder 없이 유사 문서 출력]**

In [ ]:
#Chroma dimension 관련 에러 발생 시 실행
# Chroma().delete_collection()

In [13]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 7.7 MB/s eta 0:00:00m eta 0:00:010:01:01m

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_community.document_transformers import (
    LongContextReorder,
)
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI

texts = [
    "바스켓볼은 훌륭한 스포츠입니다.",
    "플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.",
    "셀틱스는 제가 가장 좋아하는 팀입니다.",
    "이것은 보스턴 셀틱스에 관한 문서입니다."
    "저는 단순히 영화 보러 가는 것을 좋아합니다",
    "보스턴 셀틱스가 20점차로 이겼어요",
    "이것은 그냥 임의의 텍스트입니다.",
    "엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.",
    "L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.",
    "래리 버드는 상징적인 NBA 선수였습니다.",
]
# Chroma Retriever 선언(10개의 유사 문서 출력)
retriever = FAISS.from_texts(texts, OpenAIEmbeddings(model = 'text-embedding-3-small')).as_retriever(
    search_kwargs={"k": 10}
)
query = "셀틱에 대해 설명해줘"

# 유사도 기준으로 검색 결과 출력
docs = retriever.invoke(query)
for i in docs:
  print(i.page_content)

셀틱스는 제가 가장 좋아하는 팀입니다.
보스턴 셀틱스가 20점차로 이겼어요
이것은 보스턴 셀틱스에 관한 문서입니다.저는 단순히 영화 보러 가는 것을 좋아합니다
L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.
이것은 그냥 임의의 텍스트입니다.
바스켓볼은 훌륭한 스포츠입니다.
엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.
플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.
래리 버드는 상징적인 NBA 선수였습니다.


**[Long-Context Reorder 활용하여 유사 문서 출력]**

In [15]:
#LongContextReorder 선언
reordering = LongContextReorder()
#검색된 유사문서 중 관련도가 높은 문서를 맨앞과 맨뒤에 재정배치
reordered_docs = reordering.transform_documents(docs)
for i in reordered_docs:
  print(i.page_content)

셀틱스는 제가 가장 좋아하는 팀입니다.
이것은 보스턴 셀틱스에 관한 문서입니다.저는 단순히 영화 보러 가는 것을 좋아합니다
이것은 그냥 임의의 텍스트입니다.
엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.
래리 버드는 상징적인 NBA 선수였습니다.
플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.
바스켓볼은 훌륭한 스포츠입니다.
L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.
보스턴 셀틱스가 20점차로 이겼어요


### **필요없는 문서는 삭제, Contextual Compression**

In [16]:
# Helper function for printing docs
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [17]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveJsonSplitter

loader = PyPDFLoader(r"대한민국 헌법.pdf")
pages = loader.load_and_split()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(pages)
db = FAISS.from_documents(docs, OpenAIEmbeddings(model = 'text-embedding-3-small'))
retriever =db.as_retriever()

docs = retriever.invoke("대통령의 임기는?")
pretty_print_docs(docs)

Document 1:

제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
 
제71조 대통령이 궐위되거나 사고로 인하여 직무를 수행할 수 없을 때에는 국무총리, 법률이 정
한 국무위원의 순서로 그 권한을 대행한다.
 
제72조 대통령은 필요하다고 인정할 때에는 외교ㆍ국방ㆍ통일 기타 국가안위에 관한 중요정책
을 국민투표에 붙일 수 있다.
 
제73조 대통령은 조약을 체결ㆍ비준하고, 외교사절을 신임ㆍ접수 또는 파견하며, 선전포고와 강
화를 한다.
----------------------------------------------------------------------------------------------------
Document 2:

법제처                                                            11                                                       국가법령정보센터
「대한민국헌법」
③탄핵소추의 의결을 받은 자는 탄핵심판이 있을 때까지 그 권한행사가 정지된다.
④탄핵결정은 공직으로부터 파면함에 그친다. 그러나, 이에 의하여 민사상이나 형사상의 책임
이 면제되지는 아니한다.
 
                    제4장 정부
                       제1절 대통령
 
제66조 ①대통령은 국가의 원수이며, 외국에 대하여 국가를 대표한다.
②대통령은 국가의 독립ㆍ영토의 보전ㆍ국가의 계속성과 헌법을 수호할 책무를 진다.
③대통령은 조국의 평화적 통일을 위한 성실한 의무를 진다.
④행정권은 대통령을 수반으로 하는 정부에 속한다.
 
제67조 ①대통령은 국민의 보통ㆍ평등ㆍ직접ㆍ비밀선거에 의하여 선출한다.
----------------------------------------------------------------------------------------------------
Document 3:

②대통령은 국가의 안위에 관계되는 중대한

In [19]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import OpenAI

llm = ChatOpenAI(model ='gpt-4o-mini', temperature=0)
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "대통령의 임기는?"
)
pretty_print_docs(compressed_docs)

Document 1:

제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
----------------------------------------------------------------------------------------------------
Document 2:

제67조 ①대통령은 국민의 보통ㆍ평등ㆍ직접ㆍ비밀선거에 의하여 선출한다.
----------------------------------------------------------------------------------------------------
Document 3:

제68조 ①대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.


In [21]:
from langchain_classic.retrievers.document_compressors import LLMChainFilter

_filter = LLMChainFilter.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "대통령의 임기는?"
)
pretty_print_docs(compressed_docs)

Document 1:

제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
 
제71조 대통령이 궐위되거나 사고로 인하여 직무를 수행할 수 없을 때에는 국무총리, 법률이 정
한 국무위원의 순서로 그 권한을 대행한다.
 
제72조 대통령은 필요하다고 인정할 때에는 외교ㆍ국방ㆍ통일 기타 국가안위에 관한 중요정책
을 국민투표에 붙일 수 있다.
 
제73조 대통령은 조약을 체결ㆍ비준하고, 외교사절을 신임ㆍ접수 또는 파견하며, 선전포고와 강
화를 한다.
----------------------------------------------------------------------------------------------------
Document 2:

법제처                                                            11                                                       국가법령정보센터
「대한민국헌법」
③탄핵소추의 의결을 받은 자는 탄핵심판이 있을 때까지 그 권한행사가 정지된다.
④탄핵결정은 공직으로부터 파면함에 그친다. 그러나, 이에 의하여 민사상이나 형사상의 책임
이 면제되지는 아니한다.
 
                    제4장 정부
                       제1절 대통령
 
제66조 ①대통령은 국가의 원수이며, 외국에 대하여 국가를 대표한다.
②대통령은 국가의 독립ㆍ영토의 보전ㆍ국가의 계속성과 헌법을 수호할 책무를 진다.
③대통령은 조국의 평화적 통일을 위한 성실한 의무를 진다.
④행정권은 대통령을 수반으로 하는 정부에 속한다.
 
제67조 ①대통령은 국민의 보통ㆍ평등ㆍ직접ㆍ비밀선거에 의하여 선출한다.
----------------------------------------------------------------------------------------------------
Document 3:

②제1항의 선거에 있어서 최고득표자가 2

In [22]:
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter
from langchain_text_splitters import CharacterTextSplitter

llm = ChatOpenAI(model ='gpt-4o-mini', temperature=0)

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0, separator=". ")
compressor = LLMChainExtractor.from_llm(llm)
relevant_filter = EmbeddingsFilter(embeddings=OpenAIEmbeddings(model = 'text-embedding-3-small'), similarity_threshold=0.4)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, relevant_filter, compressor]
)

In [23]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "대통령의 임기는?"
)
pretty_print_docs(compressed_docs)

Document 1:

제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
----------------------------------------------------------------------------------------------------
Document 2:

제68조 ①대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.


### **가상 문서로 유사 문서 탐색, HyDE**(Hypothetical Document Embedding)

In [24]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """
당신은 LangChain, LangGraph, LangServe, LangSmith라는 LLM 기반 애플리케이션을 구축하기 위한 일련의 소프트웨어에 대한 전문가입니다.

LangChain은 LLM 애플리케이션을 구축하기 위해 쉽게 구성할 수 있는 대규모 통합 세트를 제공하는 Python 프레임워크입니다.
LangGraph는 상태 저장, 멀티 액터 LLM 애플리케이션을 쉽게 구축할 수 있는 LangChain 위에 구축된 Python 패키지입니다.
LangServe는 REST API로 LangChain 애플리케이션을 쉽게 배포할 수 있는 LangChain 위에 구축된 Python 패키지입니다.
LangSmith는 LLM 애플리케이션 추적 및 테스트를 쉽게 할 수 있는 플랫폼입니다.

사용자 질문에 최선을 다해 답변하세요. 사용자 질문에 대한 튜토리얼을 작성하는 것처럼 답변하세요.
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa_no_context = prompt | llm | StrOutputParser()

In [25]:
answer = qa_no_context.invoke(
    {
        "question": "Chain을 구축할 때 멀티모달 모델을 활용하는 방법과 Chain을 REST API로 전환하는 방법"
    }
)
print(answer)

Chain을 구축할 때 멀티모달 모델을 활용하고, 이를 REST API로 전환하는 방법에 대해 단계별로 설명하겠습니다. 이 튜토리얼에서는 LangChain과 LangServe를 사용하여 멀티모달 Chain을 구축하고 이를 REST API로 배포하는 과정을 다룹니다.

### 1단계: 멀티모달 모델 설정

멀티모달 모델은 텍스트, 이미지, 오디오 등 다양한 형태의 데이터를 처리할 수 있는 모델입니다. LangChain을 사용하여 멀티모달 Chain을 구축하기 위해 필요한 라이브러리를 설치합니다.

```bash
pip install langchain
pip install transformers
```

### 2단계: 멀티모달 Chain 구축

이제 멀티모달 Chain을 구축해보겠습니다. 예를 들어, 텍스트와 이미지를 입력으로 받아서 텍스트에 대한 설명을 생성하는 Chain을 만들어 보겠습니다.

```python
from langchain import Chain
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import SimpleSequentialChain

# LLM 설정
llm = OpenAI(model="gpt-3.5-turbo")

# 텍스트 프롬프트 템플릿
text_prompt = PromptTemplate(
    input_variables=["text", "image_description"],
    template="Given the text: '{text}' and the image description: '{image_description}', generate a detailed response."
)

# Chain 구축
text_chain = Chain(llm=llm, prompt=text_prompt)

# 멀티모달 Chain
multi_modal_chain = SimpleSequentialChain(

In [ ]:
from langchain_core.runnables import RunnablePassthrough

hyde_chain = RunnablePassthrough.assign(hypothetical_document=qa_no_context)

hyde_chain.invoke(
    {
        "question": "Chain을 구축할 때 멀티모달 모델을 활용하는 방법과 Chain을 REST API로 전환하는 방법"
    }
)